# MLflow Model Registry and Versioning

In this notebook, the production T-Learner developed in the previous notebooks is integrated with the **MLflow Model Registry**.

The Model Registry provides a structured way to manage machine learning model versions and maintain traceability between experiments, model artifacts, and deployed models.

The main objectives of this notebook are:

- Register the trained T-Learner model with MLflow.
- Create a versioned model entry.
- Add metadata and validation tags.
- Assign a meaningful model alias.
- Retrieve the registered model using its version or alias.
- Verify that the registered model can be loaded successfully.
- Validate that the registered model produces predictions.

This step moves the project from basic experiment tracking toward a more complete **MLOps model lifecycle**.

The MLflow Model Registry supports model versioning, lineage, tags, descriptions, and aliases, which can be used to identify models intended for deployment. 

In [24]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

import mlflow
import mlflow.pyfunc

from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.16.0


In [25]:
PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "criteo-research-uplift-v2.1.csv.gz"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

REPORTS_DIR = (
    PROJECT_ROOT
    / "reports"
)

MLFLOW_DIR = (
    PROJECT_ROOT
    / "mlruns"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DATA_PATH)

print("\nModels directory:")
print(MODELS_DIR)

print("\nMLflow directory:")
print(MLFLOW_DIR)

Project root:
C:\Users\ugand\customer-churn-uplift-modeling

Dataset:
C:\Users\ugand\customer-churn-uplift-modeling\data\raw\criteo-research-uplift-v2.1.csv.gz

Models directory:
C:\Users\ugand\customer-churn-uplift-modeling\models

MLflow directory:
C:\Users\ugand\customer-churn-uplift-modeling\mlruns


In [26]:
print("Dataset exists:", DATA_PATH.exists())
print("Models directory exists:", MODELS_DIR.is_dir())
print("Processed directory exists:", PROCESSED_DIR.is_dir())
print("Reports directory exists:", REPORTS_DIR.is_dir())
print("MLflow directory exists:", MLFLOW_DIR.is_dir())

Dataset exists: True
Models directory exists: True
Processed directory exists: True
Reports directory exists: True
MLflow directory exists: True


In [27]:
mlflow_tracking_path = (
    MLFLOW_DIR / "mlflow.db"
)

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_tracking_path}"
)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

MLflow tracking URI:
sqlite:///C:\Users\ugand\customer-churn-uplift-modeling\mlruns\mlflow.db


In [28]:
client = MlflowClient()

print(
    "MLflow client created successfully."
)

MLflow client created successfully.


In [29]:
EXPERIMENT_NAME = (
    "customer_uplift_modeling"
)

mlflow.set_experiment(
    EXPERIMENT_NAME
)

print("Experiment:")
print(EXPERIMENT_NAME)

Experiment:
customer_uplift_modeling


In [30]:
feature_columns = [
    f"f{i}"
    for i in range(12)
]

print("Feature columns:")
print(feature_columns)

Feature columns:
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


In [31]:
control_model_path = (
    MODELS_DIR /
    "t_learner_control_model.joblib"
)

treatment_model_path = (
    MODELS_DIR /
    "t_learner_treatment_model.joblib"
)

control_model = joblib.load(
    control_model_path
)

treatment_model = joblib.load(
    treatment_model_path
)

print("Control model:")
print(type(control_model))

print("\nTreatment model:")
print(type(treatment_model))

Control model:
<class 'sklearn.ensemble._forest.RandomForestClassifier'>

Treatment model:
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [32]:
class TLearnerUpliftModel(
    mlflow.pyfunc.PythonModel
):

    def __init__(
        self,
        control_model,
        treatment_model
    ):
        self.control_model = control_model
        self.treatment_model = treatment_model

    def predict(
        self,
        context,
        model_input
    ):
        treatment_probability = (
            self.treatment_model
            .predict_proba(model_input)[:, 1]
        )

        control_probability = (
            self.control_model
            .predict_proba(model_input)[:, 1]
        )

        uplift = (
            treatment_probability
            - control_probability
        )

        return uplift

C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [33]:
t_learner_model = TLearnerUpliftModel(
    control_model=control_model,
    treatment_model=treatment_model
)

print(
    "MLflow-compatible T-Learner created successfully."
)

MLflow-compatible T-Learner created successfully.


In [34]:
test_data = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    nrows=10
)

X_test_sample = (
    test_data[
        feature_columns
    ]
)

print(
    "Test sample shape:",
    X_test_sample.shape
)

display(
    X_test_sample.head()
)

Test sample shape: (10, 12)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679


In [35]:
test_uplift = (
    t_learner_model.predict(
        None,
        X_test_sample
    )
)

print("Sample uplift predictions:")

for i, value in enumerate(test_uplift):

    print(
        f"Row {i + 1}: {value:.6f}"
    )

Sample uplift predictions:
Row 1: -0.004079
Row 2: -0.004079
Row 3: -0.004079
Row 4: -0.004079
Row 5: -0.013535
Row 6: -0.004079
Row 7: -0.010322
Row 8: -0.004079
Row 9: -0.013535
Row 10: -0.004079


In [36]:
metrics_path = (
    REPORTS_DIR /
    "production_model_metrics.csv"
)

production_metrics = pd.read_csv(
    metrics_path
)

print("Production metrics:")

display(
    production_metrics
)

Production metrics:


,metric,value
0,ROC-AUC,0.940856
1,PR-AUC,0.178952
2,Uplift@10%,0.149753
3,Uplift@20%,0.091474
4,Mean Predicted Uplift,0.013582


In [37]:
metrics_dict = {}

for column in production_metrics.columns:

    value = (
        production_metrics[
            column
        ].iloc[0]
    )

    if pd.notna(value):

        try:
            metrics_dict[column] = float(value)

        except (
            ValueError,
            TypeError
        ):
            pass

print("Metrics to log:")

for key, value in metrics_dict.items():

    print(
        f"{key}: {value}"
    )

Metrics to log:
value: 0.940856458756722


In [38]:
model_params = {

    "model_type": "T-Learner",

    "control_model":
        "RandomForestClassifier",

    "treatment_model":
        "RandomForestClassifier",

    "feature_count": 12,

    "sample_size": 100_000,

    "test_size": 0.20,

    "random_state": 42,

    "n_estimators": 200,

    "max_depth": 10,

    "min_samples_leaf": 20,

    "class_weight": "balanced"
}

print("Model parameters:")

for key, value in model_params.items():

    print(
        f"{key}: {value}"
    )

Model parameters:
model_type: T-Learner
control_model: RandomForestClassifier
treatment_model: RandomForestClassifier
feature_count: 12
sample_size: 100000
test_size: 0.2
random_state: 42
n_estimators: 200
max_depth: 10
min_samples_leaf: 20
class_weight: balanced


In [39]:
with mlflow.start_run(
    run_name="t_learner_registry_v1"
) as run:

    run_id = run.info.run_id

    # Log parameters
    mlflow.log_params(
        model_params
    )

    # Log metrics
    mlflow.log_metrics(
        metrics_dict
    )

    # Log dataset information
    mlflow.log_param(
        "dataset",
        "Criteo Uplift v2.1"
    )

    mlflow.log_param(
        "uplift_formula",
        "P(Y|T=1,X) - P(Y|T=0,X)"
    )

    # Log model
    model_info = (
        mlflow.pyfunc.log_model(
            name="t_learner",
            python_model=t_learner_model
        )
    )

    print("Run ID:")
    print(run_id)

    print("\nModel URI:")
    print(model_info.model_uri)

    print("\nModel ID:")
    print(model_info.model_id)

2026/09/06 10:06:25 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


Run ID:
3206fbe50b624405bbe70639a2f232a7

Model URI:
models:/m-6f76c61360fa4ee3bebcc9e97966b611

Model ID:
m-6f76c61360fa4ee3bebcc9e97966b611


In [40]:
model_artifact_uri = (
    model_info.model_uri
)

print(
    "Model URI:"
)

print(
    model_artifact_uri
)

Model URI:
models:/m-6f76c61360fa4ee3bebcc9e97966b611


In [41]:
REGISTERED_MODEL_NAME = (
    "customer_uplift_t_learner"
)

registered_model = (
    mlflow.register_model(
        model_uri=model_artifact_uri,
        name=REGISTERED_MODEL_NAME
    )
)

print("Registered model:")
print(
    registered_model.name
)

print("\nModel version:")
print(
    registered_model.version
)

Registered model:
customer_uplift_t_learner

Model version:
1


Registered model 'customer_uplift_t_learner' already exists. Creating a new version of this model...
Created version '1' of model 'customer_uplift_t_learner'.


In [42]:
MODEL_VERSION = (
    registered_model.version
)

print(
    "Current model version:",
    MODEL_VERSION
)

Current model version: 1


In [43]:
client.set_model_version_tag(
    REGISTERED_MODEL_NAME,
    MODEL_VERSION,
    "model_type",
    "T-Learner"
)

client.set_model_version_tag(
    REGISTERED_MODEL_NAME,
    MODEL_VERSION,
    "dataset",
    "Criteo Uplift v2.1"
)

client.set_model_version_tag(
    REGISTERED_MODEL_NAME,
    MODEL_VERSION,
    "validation_status",
    "passed"
)

print(
    "Model version tags added successfully."
)

Model version tags added successfully.


In [44]:
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=MODEL_VERSION,
    description=(
        "T-Learner uplift model using separate "
        "Random Forest treatment and control models "
        "trained on the Criteo Uplift v2.1 benchmark."
    )
)

print(
    "Model description updated successfully."
)

Model description updated successfully.


In [45]:
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME,
    "champion",
    MODEL_VERSION
)

print(
    "Champion alias assigned to model version:",
    MODEL_VERSION
)

Champion alias assigned to model version: 1


In [46]:
champion_model = (
    client.get_model_version_by_alias(
        REGISTERED_MODEL_NAME,
        "champion"
    )
)

print("Model name:")
print(
    champion_model.name
)

print("\nVersion:")
print(
    champion_model.version
)

print("\nAliases:")
print(
    champion_model.aliases
)

print("\nTags:")
print(
    champion_model.tags
)

Model name:
customer_uplift_t_learner

Version:
1

Aliases:
['champion']

Tags:
{'model_type': 'T-Learner', 'dataset': 'Criteo Uplift v2.1', 'validation_status': 'passed'}


In [47]:
registered_models = (
    client.search_registered_models()
)

print("Registered models:\n")

for model in registered_models:

    print(
        model.name
    )

Registered models:

customer_uplift_t_learner


In [48]:
versions = (
    client.search_model_versions(
        f"name='{REGISTERED_MODEL_NAME}'"
    )
)

print(
    "Registered model versions:\n"
)

for version in versions:

    print(
        "Model:",
        version.name
    )

    print(
        "Version:",
        version.version
    )

    print(
        "Aliases:",
        version.aliases
    )

    print(
        "Tags:",
        version.tags
    )

    print("-" * 50)

Registered model versions:

Model: customer_uplift_t_learner
Version: 1
Aliases: []
Tags: {'model_type': 'T-Learner', 'dataset': 'Criteo Uplift v2.1', 'validation_status': 'passed'}
--------------------------------------------------


In [49]:
champion_uri = (
    f"models:/{REGISTERED_MODEL_NAME}@champion"
)

print(
    "Champion model URI:"
)

print(
    champion_uri
)

Champion model URI:
models:/customer_uplift_t_learner@champion


In [50]:
loaded_champion = (
    mlflow.pyfunc.load_model(
        champion_uri
    )
)

print(
    "Champion model loaded successfully."
)

print(
    type(loaded_champion)
)

Champion model loaded successfully.
<class 'mlflow.pyfunc.PyFuncModel'>


In [51]:
champion_predictions = (
    loaded_champion.predict(
        X_test_sample
    )
)

print(
    "Champion uplift predictions:"
)

for i, value in enumerate(
    champion_predictions
):

    print(
        f"Row {i + 1}: {value:.6f}"
    )

Champion uplift predictions:
Row 1: -0.004079
Row 2: -0.004079
Row 3: -0.004079
Row 4: -0.004079
Row 5: -0.013535
Row 6: -0.004079
Row 7: -0.010322
Row 8: -0.004079
Row 9: -0.013535
Row 10: -0.004079


In [52]:
registry_checks = {

    "Registered model exists":
        champion_model.name
        == REGISTERED_MODEL_NAME,

    "Model version exists":
        MODEL_VERSION is not None,

    "Champion alias assigned":
        "champion"
        in champion_model.aliases,

    "Validation status passed":
        champion_model.tags.get(
            "validation_status"
        ) == "passed",

    "Champion model loads":
        loaded_champion is not None,

    "Champion predictions generated":
        len(champion_predictions) == 10
}


print(
    "Final Model Registry Checks\n"
)

for check, status in registry_checks.items():

    print(
        f"{check}: "
        f"{'PASS' if status else 'FAIL'}"
    )

Final Model Registry Checks

Registered model exists: PASS
Model version exists: PASS
Champion alias assigned: PASS
Validation status passed: PASS
Champion model loads: PASS
Champion predictions generated: PASS


# Conclusion

In this notebook, the production T-Learner was successfully integrated with the **MLflow Model Registry**.

The major outcomes were:

- Created an MLflow-compatible T-Learner containing separate treatment and control Random Forest models.
- Validated the T-Learner using sample feature data.
- Logged the model, parameters, metrics, and dataset information to MLflow.
- Registered the T-Learner under the model name `customer_uplift_t_learner`.
- Created a versioned model entry in the MLflow Model Registry.
- Added model metadata and validation tags.
- Assigned the `champion` alias to the validated model version.
- Retrieved the champion model through the MLflow Registry.
- Successfully loaded the registered model for inference.
- Generated uplift predictions using the registered champion model.
- Completed final model registry validation checks.

The project now has a complete model management workflow:

Raw Dataset
→ Feature Engineering
→ Uplift Modeling
→ Model Evaluation
→ Production Model
→ MLflow Experiment Tracking
→ Model Registry
→ Versioned Champion Model
→ Model Loading and Inference

This establishes the foundation for the next stage of the project: **serving the registered uplift model through an API and integrating it into a production-style inference workflow**.